# 🔬 NuPheno: Pan-Cancer Nuclear Phenotyping & Detection — Kaggle Training Baseline

This self-contained notebook trains a nuclei detector on Kaggle using GPU acceleration, evaluates detection & phenotyping mAP, generates candidate predictions, and RSA-encrypts the final submission file for the **NuPheno Challenge (CV Code Bench)**.

### 📂 Dataset: [Cancer Instance Segmentation and Classification 2](https://www.kaggle.com/datasets/andrewmvd/cancer-instance-segmentation-and-classification-2)
- **Source**: Warwick TIA Centre PanNuke Dataset hosted on Kaggle by `andrewmvd`.
- **Contents**: Real $256 \times 256$ H&E histology patches across 19 human tissue types with exhaustive instance masks for 5 cell phenotypes.

### ⚡ Kaggle Setup Instructions:
1. In the right sidebar under **Input**, click **+ Add Data**.
2. In the search box, paste: `andrewmvd/cancer-instance-segmentation-and-classification-2` (or search `cancer-instance-segmentation-and-classification-2`).
3. Click the **+** button to attach it (it will mount at `/kaggle/input/cancer-instance-segmentation-and-classification-2`).
4. Under **Settings** $\rightarrow$ **Accelerator**, select **GPU T4 x2** (or GPU P100) and turn **Internet** to **On**.

## 1. Imports and Device Setup

In [ ]:
!pip install -q rsa torchvision albumentations opencv-python

import os
import sys
import glob
import json
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import scipy.ndimage as ndi
from tqdm import tqdm
import rsa

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Compute device: {device}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ Running on CPU! Enable GPU in Settings -> Accelerator -> GPU T4 for fast training.")


## 2. Load Official Kaggle Dataset (`andrewmvd/cancer-instance-segmentation-and-classification-2`)

We locate `images.npy`, `masks.npy`, and `types.npy` in `/kaggle/input/cancer-instance-segmentation-and-classification-2` and extract nuclei bounding boxes from the instance segmentation masks.

In [ ]:
CATEGORY_NAMES = {
    1: "Neoplastic",
    2: "Inflammatory",
    3: "Connective",
    4: "Dead",
    5: "Epithelial"
}
CATEGORY_COLORS = {
    1: "#E41A1C",  # Red - Neoplastic (Tumor cells)
    2: "#4DAF4A",  # Green - Inflammatory (Immune cells)
    3: "#377EB8",  # Blue - Connective (Stroma/Fibroblasts)
    4: "#FF7F00",  # Orange - Dead (Apoptotic/Necrotic)
    5: "#984EA3"   # Purple - Epithelial (Benign)
}

# Search for images.npy and masks.npy in /kaggle/input
target_dataset_slug = "cancer-instance-segmentation-and-classification-2"
image_paths = sorted(glob.glob("/kaggle/input/**/images.npy", recursive=True))
mask_paths = sorted(glob.glob("/kaggle/input/**/masks.npy", recursive=True))
type_paths = sorted(glob.glob("/kaggle/input/**/types.npy", recursive=True))

print(f"Found images.npy matches: {image_paths}")

if len(image_paths) == 0 or len(mask_paths) == 0:
    raise FileNotFoundError(
        "❌ ERROR: Could not find 'images.npy' and 'masks.npy' in /kaggle/input!\n\n"
        "Please attach the official dataset to this notebook:\n"
        "1. Click '+ Add Data' in the right-hand panel\n"
        "2. Search for: andrewmvd/cancer-instance-segmentation-and-classification-2\n"
        "3. Link: https://www.kaggle.com/datasets/andrewmvd/cancer-instance-segmentation-and-classification-2\n"
        "4. Click '+' to mount it into your notebook.\n"
    )

# Select primary dataset file
img_file = image_paths[0]
mask_file = mask_paths[0]
print(f"Loading patches from: {img_file}")
print(f"Loading masks from:   {mask_file}")

raw_images = np.load(img_file)
raw_masks = np.load(mask_file)
tissue_types = np.load(type_paths[0]) if len(type_paths) > 0 else None

# Ensure images are uint8 in range [0, 255]
if raw_images.dtype != np.uint8:
    if raw_images.max() <= 1.0:
        all_images = (raw_images * 255.0).astype(np.uint8)
    else:
        all_images = raw_images.astype(np.uint8)
else:
    all_images = raw_images

all_masks = raw_masks
print(f"Images Array Shape: {all_images.shape} (dtype: {all_images.dtype})")
print(f"Masks Array Shape:  {all_masks.shape}  (dtype: {all_masks.dtype})")
if tissue_types is not None:
    print(f"Unique Tissue Types: {np.unique(tissue_types)[:6]}... ({len(np.unique(tissue_types))} total)")

# Extract nuclei bounding boxes from real instance segmentation masks
# PanNuke mask channels (0: Neoplastic, 1: Inflammatory, 2: Connective, 3: Dead, 4: Epithelial)
sample_records = []
num_patches_to_parse = len(all_images)

print(f"Parsing nuclei bounding boxes from {num_patches_to_parse} patches using scipy find_objects...")
for i in tqdm(range(num_patches_to_parse), desc="Extracting Bounding Boxes"):
    m = all_masks[i] # (256, 256, channels)
    num_channels = min(5, m.shape[-1])
    for cat_idx in range(num_channels):
        cat_id = cat_idx + 1
        chan = m[:, :, cat_idx]
        if not np.any(chan):
            continue
        # Fast slice extraction for every instance in this channel
        slices = ndi.find_objects(chan.astype(int))
        for s in slices:
            if s is None:
                continue
            y1, y2 = s[0].start, s[0].stop
            x1, x2 = s[1].start, s[1].stop
            w = x2 - x1
            h = y2 - y1
            # Filter out 1-2 pixel noise artifacts
            if w >= 3 and h >= 3:
                sample_records.append({
                    "image_id": i,
                    "x": float(x1),
                    "y": float(y1),
                    "width": float(w),
                    "height": float(h),
                    "category_id": cat_id
                })

ann_df = pd.DataFrame(sample_records)
ann_df["area"] = ann_df["width"] * ann_df["height"]
print(f"\n✅ Successfully parsed {len(ann_df):,} real nuclei annotations across {len(all_images)} patches!")


## 2.5 Exploratory Data Analysis (EDA) on Real PanNuke Data

We visualize:
1. **Actual H&E Patches** with true bounding boxes color-coded by phenotype.
2. **Cellular Phenotype Class Distribution** (revealing clinical prevalence and class imbalance).
3. **Cellular Density Distribution** (number of nuclei per $256 \times 256$ field of view).
4. **Nucleus Area Distribution** across the 5 cell types.

In [ ]:
fig = plt.figure(figsize=(18, 10))
fig.suptitle("PanNuke Real Histopathology Dataset — Exploratory Data Analysis", fontsize=15, fontweight="bold", y=0.98)

# Top Row: 3 Sample H&E Patches with Ground-Truth Nuclei Bounding Boxes
sample_patch_ids = ann_df["image_id"].unique()[:3]
for idx, img_id in enumerate(sample_patch_ids):
    ax = fig.add_subplot(2, 3, idx + 1)
    ax.imshow(all_images[int(img_id)])
    df_p = ann_df[ann_df["image_id"] == img_id]
    for _, row in df_p.iterrows():
        cid = int(row["category_id"])
        col = CATEGORY_COLORS.get(cid, "yellow")
        rect = patches.Rectangle((row["x"], row["y"]), row["width"], row["height"],
                                 linewidth=1.8, edgecolor=col, facecolor="none")
        ax.add_patch(rect)
    tissue_str = f" ({tissue_types[int(img_id)]})" if tissue_types is not None else ""
    ax.set_title(f"Patch #{img_id}{tissue_str} - {len(df_p)} Nuclei", fontweight="bold")
    ax.axis("off")

# Legend for the 5 cellular phenotypes
legend_patches = [
    patches.Patch(color=CATEGORY_COLORS[k], label=f"{k}: {v}")
    for k, v in CATEGORY_NAMES.items()
]
fig.legend(handles=legend_patches, loc="upper center", bbox_to_anchor=(0.5, 0.93), ncol=5, frameon=True)

# Bottom-Left: Phenotype Class Distribution
ax_cls = fig.add_subplot(2, 3, 4)
cls_counts = ann_df["category_id"].value_counts().reindex(range(1, 6), fill_value=0)
cls_names = [CATEGORY_NAMES[i] for i in range(1, 6)]
cls_cols = [CATEGORY_COLORS[i] for i in range(1, 6)]
bars = ax_cls.bar(cls_names, cls_counts.values, color=cls_cols, edgecolor="black", alpha=0.85)
ax_cls.set_title("Nuclei Count by Phenotype Class", fontweight="bold")
ax_cls.set_ylabel("Annotated Nuclei Count")
ax_cls.tick_params(axis="x", rotation=20)
ax_cls.grid(axis="y", linestyle="--", alpha=0.4)
for b, cnt in zip(bars, cls_counts.values):
    pct = 100.0 * cnt / max(1, len(ann_df))
    ax_cls.text(b.get_x() + b.get_width() / 2, b.get_height() + 50, f"{cnt:,}\n({pct:.1f}%)",
                ha="center", va="bottom", fontsize=8, fontweight="bold")
ax_cls.set_ylim(0, cls_counts.max() * 1.22 if cls_counts.max() > 0 else 10)

# Bottom-Center: Nuclei Density per Patch
ax_dens = fig.add_subplot(2, 3, 5)
per_patch = ann_df.groupby("image_id").size()
ax_dens.hist(per_patch.values, bins=25, color="steelblue", edgecolor="black", alpha=0.85)
ax_dens.axvline(per_patch.mean(), color="red", linestyle="--", linewidth=2, label=f"Mean: {per_patch.mean():.1f}")
ax_dens.set_title("Cellular Density (Nuclei per Patch)", fontweight="bold")
ax_dens.set_xlabel("Nuclei per 256×256 Patch")
ax_dens.set_ylabel("Patch Count")
ax_dens.legend()
ax_dens.grid(linestyle="--", alpha=0.4)

# Bottom-Right: Bounding Box Area Distribution by Class
ax_area = fig.add_subplot(2, 3, 6)
area_groups = [ann_df[ann_df["category_id"] == i]["area"].values for i in range(1, 6)]
area_groups = [g if len(g) > 0 else np.array([0]) for g in area_groups]
bp = ax_area.boxplot(area_groups, tick_labels=cls_names, patch_artist=True, showfliers=False)
for patch, col in zip(bp["boxes"], cls_cols):
    patch.set_facecolor(col)
    patch.set_alpha(0.75)
ax_area.set_title("Nucleus Bounding Box Area by Phenotype", fontweight="bold")
ax_area.set_ylabel("Area (pixels²)")
ax_area.tick_params(axis="x", rotation=20)
ax_area.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout(rect=[0, 0, 1, 0.89])
plt.show()


## 3. PyTorch Dataset & DataLoaders

We split the real PanNuke patches into **Train (70%)**, **Validation (15%)**, and **Test (15%)** sets.

In [ ]:
class PanNukeDetectionDataset(Dataset):
    def __init__(self, images, annotations_df, image_ids):
        self.images = images
        self.annotations = annotations_df
        self.image_ids = list(image_ids)
        
    def __len__(self):
        return len(self.image_ids)
        
    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img = self.images[img_id].astype(np.float32) / 255.0
        # Transpose HWC -> CHW
        img_tensor = torch.tensor(img, dtype=torch.float32).permute(2, 0, 1)
        
        df_boxes = self.annotations[self.annotations["image_id"] == img_id]
        boxes = []
        labels = []
        
        for _, row in df_boxes.iterrows():
            x1 = row["x"]
            y1 = row["y"]
            x2 = x1 + row["width"]
            y2 = y1 + row["height"]
            if x2 > x1 + 1 and y2 > y1 + 1:
                boxes.append([x1, y1, x2, y2])
                labels.append(int(row["category_id"]))
                
        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)
            
        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([img_id])
        }
        return img_tensor, target

def collate_fn(batch):
    return tuple(zip(*batch))

# Split into Train (70%), Validation (15%), and Test (15%)
total_imgs = len(all_images)
indices = np.arange(total_imgs)
n_train = int(0.70 * total_imgs)
n_val = int(0.15 * total_imgs)

train_idx = indices[:n_train]
val_idx = indices[n_train:n_train + n_val]
test_idx = indices[n_train + n_val:]

train_dataset = PanNukeDetectionDataset(all_images, ann_df, train_idx)
val_dataset = PanNukeDetectionDataset(all_images, ann_df, val_idx)
test_dataset = PanNukeDetectionDataset(all_images, ann_df, test_idx)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)

print(f"Patches Split: {len(train_dataset)} Train | {len(val_dataset)} Validation | {len(test_dataset)} Test")


## 4. Build Model & Training Loop

We fine-tune a pre-trained `fasterrcnn_mobilenet_v3_large_fpn` detector for our 5 nuclei classes (+ background).
During training, we track **Train Loss** and compute **Validation Loss** using `torch.no_grad()` at each epoch.

In [ ]:
def get_model(num_classes=6):
    model = fasterrcnn_mobilenet_v3_large_fpn(weights="DEFAULT")
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

model = get_model(num_classes=6).to(device)
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(params, lr=1e-4, weight_decay=1e-4)

# Set EPOCHS = 3 for initial baseline (or increase to 15-30 for maximum competition performance)
EPOCHS = 3
print(f"Training Faster R-CNN on {device} for {EPOCHS} epochs...")

history = {"train_loss": [], "val_loss": []}
for epoch in range(1, EPOCHS + 1):
    # ==================== 1. TRAINING LOOP ====================
    model.train()
    total_train_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]")
    for images, targets in pbar:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        
        total_train_loss += losses.item()
        pbar.set_postfix({"train_loss": f"{losses.item():.4f}"})
        
    avg_train_loss = total_train_loss / len(train_loader)
    history["train_loss"].append(avg_train_loss)

    # ================== 2. VALIDATION LOSS LOOP ==================
    total_val_loss = 0.0
    with torch.no_grad():
        for images, targets in tqdm(val_loader, desc=f"Epoch {epoch}/{EPOCHS} [Val]", leave=False):
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            total_val_loss += losses.item()
            
    avg_val_loss = total_val_loss / len(val_loader)
    history["val_loss"].append(avg_val_loss)
    
    print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] -> Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

torch.save(model.state_dict(), "/kaggle/working/best_fasterrcnn_pannuke.pt")
print("✅ Saved trained model weights to /kaggle/working/best_fasterrcnn_pannuke.pt")


## 5. View Evaluation Metrics (mAP@[0.50:0.95] & Class Breakdown)

We evaluate the validation split and calculate:
- **mAP@[0.50:0.95]** (Official competition metric)
- **mAP@0.50**
- **Per-Class AP** for Neoplastic, Inflammatory, Connective, Dead, and Epithelial cells

In [ ]:
def box_iou_xywh(box1, box2):
    x1, y1, w1, h1 = box1
    x2, y2, w2, h2 = box2
    xi1, yi1 = max(x1, x2), max(y1, y2)
    xi2, yi2 = min(x1 + w1, x2 + w2), min(y1 + h1, y2 + h2)
    inter_w, inter_h = max(0.0, xi2 - xi1), max(0.0, yi2 - yi1)
    intersection = inter_w * inter_h
    union = (w1 * h1) + (w2 * h2) - intersection
    return float(intersection / union) if union > 0 else 0.0

def compute_ap(gt_boxes_by_img, pred_boxes_all, iou_thresh=0.50):
    total_gt = sum(len(b) for b in gt_boxes_by_img.values())
    if total_gt == 0 or len(pred_boxes_all) == 0:
        return 0.0
    
    sorted_preds = sorted(pred_boxes_all, key=lambda x: x["score"], reverse=True)
    matched_gt = {img_id: set() for img_id in gt_boxes_by_img}
    tp = np.zeros(len(sorted_preds))
    fp = np.zeros(len(sorted_preds))
    
    for idx, pred in enumerate(sorted_preds):
        img_id = pred["image_id"]
        pred_box = pred["bbox"]
        gt_boxes = gt_boxes_by_img.get(img_id, [])
        
        best_iou, best_idx = 0.0, -1
        for g_idx, g_box in enumerate(gt_boxes):
            iou = box_iou_xywh(pred_box, g_box)
            if iou > best_iou:
                best_iou, best_idx = iou, g_idx
                
        if best_iou >= iou_thresh and best_idx not in matched_gt[img_id]:
            tp[idx] = 1
            matched_gt[img_id].add(best_idx)
        else:
            fp[idx] = 1
            
    cum_tp = np.cumsum(tp)
    cum_fp = np.cumsum(fp)
    recalls = cum_tp / float(total_gt)
    precisions = cum_tp / np.maximum(cum_tp + cum_fp, 1e-12)
    
    # 101-point COCO interpolation
    rec_thresholds = np.linspace(0.0, 1.0, 101)
    p_rec = []
    for r in rec_thresholds:
        cand = precisions[recalls >= r]
        p_rec.append(np.max(cand) if len(cand) > 0 else 0.0)
    return float(np.mean(p_rec))

# Run evaluation on Validation Split
model.eval()
val_predictions = []
val_gt_by_cat = {cat_id: {} for cat_id in range(1, 6)}

with torch.no_grad():
    for images, targets in tqdm(val_loader, desc="Evaluating Validation Set"):
        images = [img.to(device) for img in images]
        preds = model(images)
        
        for target, pred in zip(targets, preds):
            img_id = int(target["image_id"].item())
            
            # Record ground truth
            gt_boxes = target["boxes"].cpu().numpy()
            gt_labels = target["labels"].cpu().numpy()
            for b, l in zip(gt_boxes, gt_labels):
                cid = int(l)
                if cid in val_gt_by_cat:
                    if img_id not in val_gt_by_cat[cid]:
                        val_gt_by_cat[cid][img_id] = []
                    val_gt_by_cat[cid][img_id].append([float(b[0]), float(b[1]), float(b[2]-b[0]), float(b[3]-b[1])])
                    
            # Record predictions
            p_boxes = pred["boxes"].cpu().numpy()
            p_labels = pred["labels"].cpu().numpy()
            p_scores = pred["scores"].cpu().numpy()
            
            for b, l, s in zip(p_boxes, p_labels, p_scores):
                if s >= 0.20:
                    val_predictions.append({
                        "image_id": img_id,
                        "category_id": int(l),
                        "bbox": [float(b[0]), float(b[1]), float(b[2]-b[0]), float(b[3]-b[1])],
                        "score": float(s)
                    })

# Compute per-class and overall mAP
iou_thresholds = np.linspace(0.50, 0.95, 10)
metric_summary = []

for cid in range(1, 6):
    name = CATEGORY_NAMES[cid]
    c_preds = [p for p in val_predictions if p["category_id"] == cid]
    n_gt = sum(len(boxes) for boxes in val_gt_by_cat[cid].values())
    
    if n_gt == 0:
        continue
        
    ap_50 = compute_ap(val_gt_by_cat[cid], c_preds, iou_thresh=0.50)
    ap_range = np.mean([compute_ap(val_gt_by_cat[cid], c_preds, iou_thresh=iou) for iou in iou_thresholds])
    
    metric_summary.append({
        "Category": f"{cid}. {name}",
        "GT Cells": n_gt,
        "Predictions": len(c_preds),
        "AP@0.50": round(ap_50, 4),
        "mAP@[0.50:0.95]": round(float(ap_range), 4)
    })

df_metrics = pd.DataFrame(metric_summary)
overall_map50 = df_metrics["AP@0.50"].mean() if len(df_metrics) > 0 else 0.0
overall_map = df_metrics["mAP@[0.50:0.95]"].mean() if len(df_metrics) > 0 else 0.0

print("=" * 65)
print("🏆 VALIDATION METRICS BREAKDOWN (REAL PANNUKE DATA):")
print("=" * 65)
print(df_metrics.to_string(index=False))
print("-" * 65)
print(f"🌟 Overall mAP@0.50:        {overall_map50:.4f}")
print(f"🥇 Primary mAP@[0.50:0.95]: {overall_map:.4f}  <-- OFFICIAL SCORE")
print("=" * 65)


### Visual Verification: Ground Truth vs Model Predictions

In [ ]:
sample_img, sample_target = val_dataset[0]
model.eval()
with torch.no_grad():
    pred = model([sample_img.to(device)])[0]

img_np = (sample_img.permute(1, 2, 0).cpu().numpy() * 255.0).astype(np.uint8)
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# Ground Truth
axes[0].imshow(img_np)
axes[0].set_title("Real Ground Truth Nuclei", fontsize=13, fontweight="bold")
axes[0].axis("off")
for box, label in zip(sample_target["boxes"].numpy(), sample_target["labels"].numpy()):
    x1, y1, x2, y2 = box
    col = CATEGORY_COLORS.get(int(label), "yellow")
    rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor=col, facecolor="none")
    axes[0].add_patch(rect)

# Model Predictions
axes[1].imshow(img_np)
axes[1].set_title(f"Model Predictions (mAP@[0.50:0.95]: {overall_map:.3f})", fontsize=13, fontweight="bold")
axes[1].axis("off")
p_boxes = pred["boxes"].cpu().numpy()
p_labels = pred["labels"].cpu().numpy()
p_scores = pred["scores"].cpu().numpy()

for box, label, score in zip(p_boxes, p_labels, p_scores):
    if score >= 0.35:
        x1, y1, x2, y2 = box
        col = CATEGORY_COLORS.get(int(label), "cyan")
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor=col, facecolor="none")
        axes[1].add_patch(rect)
        axes[1].text(x1, max(0, y1 - 3), f"{CATEGORY_NAMES.get(int(label), 'Cell')} {score:.2f}",
                     color=col, fontsize=8, backgroundcolor="black")

plt.tight_layout()
plt.show()


## 6. Test Inference & Submission Generation

We evaluate the test patches, extract predicted bounding boxes `[x, y, width, height]`, phenotype `category_id`, and confidence `score`.

In [ ]:
model.eval()
submission_rows = []

with torch.no_grad():
    for images, targets in tqdm(test_loader, desc="Running Inference on Test Set"):
        images = [img.to(device) for img in images]
        predictions = model(images)
        
        for target, pred in zip(targets, predictions):
            img_id = int(target["image_id"].item())
            boxes = pred["boxes"].cpu().numpy()
            labels = pred["labels"].cpu().numpy()
            scores = pred["scores"].cpu().numpy()
            
            # Filter by confidence threshold
            keep = scores >= 0.30
            for box, label, score in zip(boxes[keep], labels[keep], scores[keep]):
                x1, y1, x2, y2 = box
                submission_rows.append({
                    "image_id": img_id,
                    "category_id": int(label),
                    "x": round(float(x1), 1),
                    "y": round(float(y1), 1),
                    "width": round(float(x2 - x1), 1),
                    "height": round(float(y2 - y1), 1),
                    "score": round(float(score), 4)
                })

sub_df = pd.DataFrame(submission_rows)
csv_out_path = "/kaggle/working/my_team_submission.csv"
sub_df.to_csv(csv_out_path, index=False)
print(f"Generated submission CSV with {len(sub_df)} nuclei detections at {csv_out_path}")
sub_df.head(10)


## 7. Official RSA-2048 Encryption for GitHub PR

Participants submit their predictions encrypted using the competition RSA public key.
This prevents test-set cheating and lets the automated GitHub Actions grader decrypt and score the PR securely.

In [ ]:
# Official 2048-bit Public Key for PanNuke Challenge
PUBLIC_KEY_PEM = """-----BEGIN RSA PUBLIC KEY-----
MIIBCgKCAQEAuukYzCRwJ9NpBG9eU3Mhd/zlw2oPYCavrIHBiQ5W11xxxWjWpKNf
QA3ofeNKFCn8jUgemV8aZALoS3TQqaf512Fule0EpR9L92QyS4uZhQYjJxWyqhOK
msfPBpMyCv6TH/3TtEaCxKgWXMDfWD8YWBbIu9/ksKSy6Cv/Fb8pRnkOXd8QN6aA
l3HNUjbTAqogJcm+05UrzgmNiU6ci8QXYSbwpGO6glQS67242R+ru9mJ5TfLyy3e
8NFdm7LubfaSZLrPOSqDAOzhylESfXuI/F261nYxWiFkSlE+t+ZOq+/62ZhP1d+J
470eoQDCu52QYIv+8epZoP2114q8sec9swIDAQAB
-----END RSA PUBLIC KEY-----"""

def encrypt_submission_file(csv_path, enc_output_path):
    pub_key = rsa.PublicKey.load_pkcs1(PUBLIC_KEY_PEM.encode("utf-8"))
    
    with open(csv_path, "rb") as f:
        raw_bytes = f.read()
        
    chunk_size = 245  # 245 bytes for RSA-2048 PKCS#1 v1.5
    encrypted_data = b""
    
    for i in range(0, len(raw_bytes), chunk_size):
        chunk = raw_bytes[i:i + chunk_size]
        encrypted_data += rsa.encrypt(chunk, pub_key)
        
    with open(enc_output_path, "wb") as f:
        f.write(encrypted_data)
        
    print(f"✅ Successfully encrypted submission to: {enc_output_path}")
    print(f"Encrypted size: {len(encrypted_data):,} bytes (from {len(raw_bytes):,} raw bytes)")

enc_out = "/kaggle/working/my_team_submission.csv.enc"
encrypt_submission_file(csv_out_path, enc_out)


## 8. Submission Instructions for GitHub

1. In the Kaggle notebook, open the right sidebar **Output** section.
2. Download `my_team_submission.csv.enc`.
3. Rename the file to your team's name, e.g. `<your_team_name>.csv.enc`.
4. Place the file inside `challenge_01_pannuke_detection/submissions/<your_team_name>.csv.enc`.
5. Open a Pull Request to the GitHub repository.
6. The automated GitHub Action grader will automatically decrypt your submission, compute mAP@[0.50:0.95], and comment on your PR!